In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import os

# --- 1. Load the Model ---
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model_path = "models/FasterRCNN_ResNet50_Final.pth"


model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
num_ftrs = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(num_ftrs, 2) # 2 classes: Background + Seed

# Load weights
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("Best weights loaded successfully.")
else:
    print(f"ERROR: Model file not found at {model_path}")

model.to(device)
model.eval()


In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torchvision.ops import nms
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2


# --- 2. Setup Inference Parameters ---
test_folder = "../data/test-images/maize-test"
CONF_THRESHOLD = 0.90
NMS_THRESHOLD = 0.3  

# Match your exact training normalization and size
test_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()        
])

# --- 3. Process Images ---
test_images = [f for f in os.listdir(test_folder) if f.endswith(('.jpg', '.png', '.jpeg'))][:]

if not test_images:
    print("No images found in the test folder. Check your path!")

plt.figure(figsize=(20, 10))

for img_name in test_images:
    img_path = os.path.join(test_folder, img_name)
    
    # Read image
    bgr_img = cv2.imread(img_path)
    rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    orig_h, orig_w, _ = rgb_img.shape
    
    # Pre-process
    transformed = test_transform(image=rgb_img)
    img_tensor = transformed['image'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        prediction = model(img_tensor)[0]
        print(prediction)
    
    boxes = prediction['boxes']
    scores = prediction['scores']
    
    keep = nms(boxes, scores, NMS_THRESHOLD)
    boxes = boxes[keep].cpu().numpy()
    scores = scores[keep].cpu().numpy()
    
    count = 0
    for box, score in zip(boxes, scores):
        if score > CONF_THRESHOLD:
            count += 1
            x1, y1, x2, y2 = box
            x1, x2 = int(x1 * (orig_w / 224)), int(x2 * (orig_w / 224))
            y1, y2 = int(y1 * (orig_h / 224)), int(y2 * (orig_h / 224))
            
            cv2.rectangle(rgb_img, (x1, y1), (x2, y2), (0, 255, 0), 4)
            cv2.putText(rgb_img, f'{score:.2f}', (x1, y1-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

    # --- PRINT ONE BY ONE ---
    plt.figure(figsize=(12, 8)) # Larger size for better viewing
    plt.imshow(rgb_img)
    plt.title(f"Image: {img_name} | Seeds Detected: {count}", fontsize=15)
    plt.axis('off')
    plt.show() # This forces the notebook to print this image before starting the next loop


# For Feeding Anothe model

In [ ]:
import os
import torch
import cv2
import numpy as np

# Settings for the proposal stage
CROP_CONF_THRESHOLD = 0.95  # Higher threshold for cropping to avoid "junk" crops
NMS_THRESHOLD = 0.3
IMAGE_SIZE = 224

all_crops_data = [] # List to store metadata for the next block

model.eval()
with torch.no_grad():
    for img_name in test_images:
        img_path = os.path.join(test_folder, img_name)
        bgr_img = cv2.imread(img_path)
        rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
        h, w, _ = rgb_img.shape
        
        # Prep tensor
        transformed = test_transform(image=rgb_img)
        img_tensor = transformed['image'].unsqueeze(0).to(device)
        
        # Predict
        prediction = model(img_tensor)[0]
        
        # Apply NMS
        keep = nms(prediction['boxes'], prediction['scores'], NMS_THRESHOLD)
        boxes = prediction['boxes'][keep].cpu().numpy()
        scores = prediction['scores'][keep].cpu().numpy()
        
        for box, score in zip(boxes, scores):
            if score > CROP_CONF_THRESHOLD:
                # Scale back to original high-res size
                x1 = int(box[0] * (w / IMAGE_SIZE))
                y1 = int(box[1] * (h / IMAGE_SIZE))
                x2 = int(box[2] * (w / IMAGE_SIZE))
                y2 = int(box[3] * (h / IMAGE_SIZE))
                
                # Clip coordinates to stay inside image boundaries
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                
                all_crops_data.append({
                    'parent_image': img_name,
                    'coords': (x1, y1, x2, y2),
                    'full_image_rgb': rgb_img # Keeping ref for cropping
                })

print(f"Extraction complete. Found {len(all_crops_data)} candidate seeds for cropping.")


In [ ]:
# Create directory for the crops
output_crop_dir = "../data/seed_test"
os.makedirs(output_crop_dir, exist_ok=True)

cropped_images_paths = []

for i, crop_info in enumerate(all_crops_data):
    x1, y1, x2, y2 = crop_info['coords']
    full_img = crop_info['full_image_rgb']
    
    # The actual crop (NumPy slicing: [y:y, x:x])
    seed_crop = full_img[y1:y2, x1:x2]
    
    # Check if crop is valid (not empty)
    if seed_crop.size > 0:
        crop_filename = f"crop_{i}_{crop_info['parent_image']}"
        crop_save_path = os.path.join(output_crop_dir, crop_filename)
        
        # Save as BGR for OpenCV
        cv2.imwrite(crop_save_path, cv2.cvtColor(seed_crop, cv2.COLOR_RGB2BGR))
        cropped_images_paths.append(crop_save_path)

print(f"Saved {len(cropped_images_paths)} individual seed images to {output_crop_dir}")


# Testing Purposes

In [ ]:
import matplotlib.pyplot as plt

# 1. Get the list of saved crop files
all_saved_crops = sorted([os.path.join(output_crop_dir, f) for f in os.listdir(output_crop_dir)])

if len(all_saved_crops) < 10:
    print(f"Note: Only {len(all_saved_crops)} crops found. Printing all of them.")
    subset_to_show = all_saved_crops
else:
    # Pick first 5 and last 5
    subset_to_show = all_saved_crops[:5] + all_saved_crops[-5:]

# 2. Setup the plot (2 rows, 5 columns)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

print(f"Displaying the first 5 (top row) and last 5 (bottom row) crops from {output_crop_dir}...")

for i, img_path in enumerate(subset_to_show):
    # Load the crop
    crop_img = cv2.imread(img_path)
    crop_img = cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB)
    
    # Show in grid
    axes[i].imshow(crop_img)
    
    # Label them "First" or "Last"
    label = f"First #{i+1}" if i < 5 else f"Last #{i-4}"
    axes[i].set_title(f"{label}\nSize: {crop_img.shape[1]}x{crop_img.shape[0]}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms 

# A simple Dataset class to load the crops we just made
class SeedCropDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB") # PIL is better for torchvision transforms
        if self.transform:
            img = self.transform(img) # No ['image'] index here
        return img

# Define transforms for your SECOND model (e.g., ResNet classifier)

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# Create DataLoader
crop_dataset = SeedCropDataset(cropped_images_paths, transform=test_transform)
test_loader = DataLoader(crop_dataset, batch_size=32, shuffle=False, num_workers = 4)

print("Batch loader ready. You can now feed 'crop_loader' into your classification model.")

In [ ]:
import torch 
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

ResNet50_path = '../models/ResNet50_maize_seeds_NEW.pth'


In [ ]:
model = models.resnet50(weights=None) 


In [ ]:
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 1),
    nn.Sigmoid()
)

model.load_state_dict(torch.load(ResNet50_path ))
print("Weights loaded successfully")

device = torch.device("cuda")
model.to(device)

In [ ]:
import math
import matplotlib.pyplot as plt
import cv2
import pandas as pd
from tqdm import tqdm

model.eval()
predictions_list = []
all_paths = cropped_images_paths[:]  # Every single crop found

# 1. Run Inference on everything
print(f" Processing {len(all_paths)} seeds...")
with torch.no_grad():
    # 1. Loop through batches (32 images at a time)
    for batch_idx, img_tensors in enumerate(tqdm(test_loader)):
        img_tensors = img_tensors.to(device)
        
        # 2. Forward Pass (returns a tensor of 32 probabilities)
        outputs = model(img_tensors) 
        
        # 3. Loop through each individual result in that batch
        for i in range(outputs.size(0)):
            # Get the single probability for this specific image
            prob = outputs[i].item() 
            
            # Your mapping logic (0=Good, 1=Bad)
            label = "Bad" if prob > 0.9 else "Good"
            
            # Calculate the global index to find the correct path in all_paths
            global_idx = batch_idx * test_loader.batch_size + i
            
            # Safety check: make sure we don't go past our paths list
            if global_idx < len(all_paths):
                predictions_list.append({
                    "path": all_paths[global_idx],
                    "prob": prob,
                    "label": label
                })
# 2. Setup Visualization for ALL (or a large subset)
# Change 'max_images' to len(all_paths) if you want literally every single one
max_images = len(all_paths) 
cols = 5
rows = math.ceil(max_images / cols)

plt.figure(figsize=(20, 4 * rows))

for i in range(max_images):
    # Load and prep image
    img_path = predictions_list[i]['path']
    prob = predictions_list[i]['prob']
    label = predictions_list[i]['label']
    
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Styling
    color = 'red' if label == "Bad" else 'green'
    
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    # Removed "Raw", simplified to Pred and Confidence %
    plt.title(f"Pred: {label}", color=color, fontsize=12)
    plt.axis('off')

plt.tight_layout()
plt.show()

# 3. Final Summary Report
results_df = pd.DataFrame(predictions_list)
print("\n" + "="*30)
print("      FINAL QUALITY SUMMARY")
print("="*30)
print(results_df['label'].value_counts())
print("-" * 30)
good_pct = (results_df['label'] == 'Good').mean() * 100
print(f"Overall Quality Score: {good_pct:.2f}% Good")
print("="*30)